# 01 — One Sentence In. Production Prompt Out.

**The pitch.** You type a task description. `PromptArchitect.build()` generates a complete 9-section production prompt — role, goal, binding rules, style, reasoning strategy, examples, output contract, guard rails, and the task itself — structured according to research-backed positioning principles. What takes a prompt engineer two hours takes this one call.

**What you'll see in this notebook:**
- How a plain English task becomes a 1 500-token structured prompt automatically
- The 9-section architecture and the research behind each zone's position
- The 5 atomic thinking strategies and 5 named archetypes — and how to pick them
- How combining strategies changes output behavior
- Provider-aware rendering (the same prompt, differently structured for OpenAI vs Anthropic vs Gemini)

**Not covered here:** building manually and improving an existing prompt — that's notebook 02.

---
> **Research grounding.** The 9-section ordering is not arbitrary. It follows position-sensitivity findings from LLM attention research:
> - **Primacy zone** (Role, Goal first): models weight early tokens more heavily — *Liu et al. 2023, "Lost in the Middle"*
> - **Recency zone** (Task always last): the final instruction before generation receives the highest recall — *Li et al. 2023*
> - **Middle** (Examples only in the assembled prompt): few-shot calibration before the output contract — *Li et al. 2025*
> - **Recency-adjacent** (Reasoning, since 0.10.1): thinking-strategy block is rendered **just before YOUR TASK**, not in the middle, so it is not lost in long-context prompts
> - **CO-STAR framework** (Output Contract, Guard Rails in late position): validated by Singapore GovTech prompt engineering team
> - **Thinking strategies** map directly to published techniques: Chain of Thought *(Wei et al. 2022)*, Tree of Thought *(Yao et al. 2023)*, Self-Reflection *(Shinn et al. 2023 — Reflexion)*

## Install and setup

In [4]:
#%pip install -q -U "mycontext-ai>=0.10.2"
import os
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

from dotenv import load_dotenv
from IPython.display import display_markdown

project_root = Path.cwd()
# adjust as needed
env_path = project_root / "content-series" / ".env"


load_dotenv(env_path)

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")

def md(s):
    display_markdown(s,raw=True)

import mycontext


md(f"**mycontext-ai** `{getattr(mycontext, '__version__', '?')}`")




**mycontext-ai** `0.10.2`

## Part 1 — The magic: one sentence → 9-section prompt

We start with the most dramatic demo: the shortest possible task string and watch what comes out.

In [5]:
from mycontext.intelligence import PromptArchitect

task = "Analyze customer support tickets"

architect = PromptArchitect(provider="openai")
result = architect.build(task)
prompt = result.improved_context.assemble()
md(
    f"**Input:** {len(task)} characters  ·  **Output:** {len(prompt)} characters ({len(prompt.split())} words)\n\n"
    f"```\n{prompt}\n```",
)


**Input:** 32 characters  ·  **Output:** 2327 characters (354 words)

```
## ROLE

You are a senior data analyst with 10 years of experience in customer support ticket analysis. Scope: Limit to ticket analysis. Do not drift into unrelated topics.

## GOAL

**Your mission:** Identify every recurring issue in customer support tickets — accomplish this fully.

## RULES

**You MUST follow these rules at all times:**
  1. Every analysis must include ticket ID, issue description, and frequency of occurrence.
  2. Every claim must be grounded in the provided ticket data. If absent, state: 'Not found in the provided material.'
  3. Every finding must include at least two supporting examples from the ticket data.
  4. Every identified issue must be categorized by severity level.
  5. Every analysis must be completed within the specified timeframe.

## STYLE

**Tone & voice:** Formal, concise, technical. Omit jargon without definition and preamble or greeting.

## EXAMPLES

Learn from these examples of expected input → output:

**Example 1:**
Input: Customer support ticket data showing various issues reported by users.
Output: KEY FINDING: High frequency of login issues. EVIDENCE: - Ticket ID 123: 15 occurrences - Ticket ID 456: 10 occurrences - Ticket ID 789: 8 occurrences. RECOMMENDED ACTION: Investigate the login system for potential bugs.

**Example 2:**
Input: Customer support ticket data with no recurring issues.
Output: KEY FINDING: No recurring issues identified. EVIDENCE: - Ticket ID 101: Unique issue - Ticket ID 102: Unique issue. RECOMMENDED ACTION: Continue monitoring for future trends.

## OUTPUT FORMAT

Return ONLY a 3-part structure: 1) KEY FINDING (1 sentence), 2) EVIDENCE (2-3 bullets), 3) RECOMMENDED ACTION (1 sentence).

## GUARD RAILS

Must NOT include:
  - Omit speculation.
  - Omit hedging language: probably, might, could be.

## ANALYTICAL AND REPORTING APPROACH

Apply in order: **step_by_step → verify**
  - **Chain of Thought**: Think through this step by step. Break the problem down into stages, show your reasoning at each stage, then give your final answer.
  - **Self-Reflection**: After providing your answer, critically review it. Check for errors, missing information, unsupported claims, or logical gaps. If you find issues, revise your answer.

---

## YOUR TASK

Analyze the provided customer support ticket data to identify recurring issues.
```

## Part 2 — The 9 sections: what each one does and why it's in that position

The architecture isn't a list of best practices — it's a **position-sensitive system**. Section order matters because LLM attention is not uniform across a prompt.

In [6]:
from mycontext.intelligence.prompt_architect import SECTION_NAMES

md(f"**JSON / schema:** `{' → '.join(SECTION_NAMES)}`")

# Matches src/mycontext/core.py::_assemble_research_flow
assembly_guide = [
    ("role", "① PRIMACY", "Persona anchor — early tokens weighted heavily (Liu et al. 2023)."),
    ("goal", "② PRIMACY", "Mission immediately after role."),
    ("rules", "③ INSTRUCTIONS", "Binding must/never constraints."),
    ("style", "④ INSTRUCTIONS", "Tone, register, formality."),
    ("examples", "⑤ MIDDLE", "Few-shot input→output before the format contract."),
    ("(knowledge)", "⑥ OPTIONAL", "Inserted here if `Context.knowledge` is set."),
    ("output_contract", "⑦ LATE", "## OUTPUT FORMAT — CO-STAR late band."),
    ("guard_rails", "⑧ LATE", "Omit rules and uncertainty handling."),
    ("reasoning", "⑧.5 RECENCY", "Thinking strategy block immediately before YOUR TASK (not middle — avoids loss in long prompts)."),
    ("task", "⑨ RECENCY", "YOUR TASK always last — highest recall at generation (Li et al. 2023)."),
]

md(f"{'Assembled section':<20} {'Slot':<14} {'Why here?'}")
md("-" * 88)
for name, slot, why in assembly_guide:
    md(f"{name:<20} {slot:<14} {why}")


**JSON / schema:** `role → goal → rules → style → reasoning → examples → output_contract → guard_rails → task`

Assembled section    Slot           Why here?

----------------------------------------------------------------------------------------

role                 ① PRIMACY      Persona anchor — early tokens weighted heavily (Liu et al. 2023).

goal                 ② PRIMACY      Mission immediately after role.

...

(knowledge)          ⑥ OPTIONAL     Inserted here if `Context.knowledge` is set.

output_contract      ⑦ LATE         ## OUTPUT FORMAT — CO-STAR late band.

guard_rails          ⑧ LATE         Omit rules and uncertainty handling.

reasoning            ⑧.5 RECENCY    Thinking strategy block immediately before YOUR TASK (not middle — avoids loss in long prompts).

task                 ⑨ RECENCY      YOUR TASK always last — highest recall at generation (Li et al. 2023).

## Part 3 — The 5 atomic thinking strategies

Each strategy maps to a **reasoning injection** rendered in the **recency zone** (just before **YOUR TASK**) when `PromptArchitect` builds a `Context` — so it stays salient in long prompts. The LLM applies that approach when it executes the task.

In [3]:
from mycontext import THINKING_STRATEGIES

md("Atomic thinking strategies:\n")
for slug, (label, injection) in THINKING_STRATEGIES.items():
    md(f"  {slug:<20} → {label}")
    md(f"    Injection: \"{injection[:120]}...\"")


Atomic thinking strategies:


  step_by_step         → Chain of Thought

    Injection: "Think through this step by step. Break the problem down into stages, show your reasoning at each stage, then give your f..."

  multiple_angles      → Tree of Thought

    Injection: "Before answering, brainstorm at least 3 different approaches or perspectives. Briefly evaluate the strengths and weaknes..."

...

    Injection: "After providing your answer, critically review it. Check for errors, missing information, unsupported claims, or logical..."

  explain_simply       → Simplification

    Injection: "Explain your reasoning in simple, everyday language that anyone can understand. Avoid jargon and technical terms. Use an..."

  creative             → Divergent Thinking

    Injection: "Explore unconventional, surprising, and creative ideas. Don't limit yourself to the obvious answer. Challenge assumption..."

In [4]:
# Each strategy maps directly to a published technique
research_map = {
    "step_by_step":  "Chain of Thought — Wei et al. 2022 (NeurIPS)",
    "multiple_angles": "Tree of Thought — Yao et al. 2023",
    "verify":        "Self-Reflection / Reflexion — Shinn et al. 2023",
    "explain_simply": "Socratic simplification — general pedagogy literature",
    "creative":      "Divergent thinking — Guilford 1967; re-validated by Girotra et al. 2023 (GPT-4 creativity)",
}

md(f"{'Strategy slug':<22} {'Research basis'}")
md("-" * 80)
for slug, ref in research_map.items():
    md(f"{slug:<22} {ref}")


Strategy slug          Research basis

--------------------------------------------------------------------------------

step_by_step           Chain of Thought — Wei et al. 2022 (NeurIPS)

multiple_angles        Tree of Thought — Yao et al. 2023

verify                 Self-Reflection / Reflexion — Shinn et al. 2023

explain_simply         Socratic simplification — general pedagogy literature

creative               Divergent thinking — Guilford 1967; re-validated by Girotra et al. 2023 (GPT-4 creativity)

## Part 4 — The 5 named archetypes: pre-validated strategy combinations

Archetypes are **validated compositions** of atomic strategies for common task types. Instead of guessing which strategies to combine, you pick the archetype that fits your task.

In [7]:
from mycontext.intelligence.prompt_architect import REASONING_ARCHETYPES, REASONING_STRATEGY_CHOICES

md("Named archetypes (pre-validated strategy compositions):\n")
archetype_use_cases = {
    "analytical":    "Data analysis, structured problem-solving, reports",
    "deliberative":  "Strategy decisions, trade-off analysis, policy questions",
    "explanatory":   "Documentation, tutorials, customer-facing explanations",
    "creative":      "Brainstorming, ideation, copywriting, scenario planning",
    "high_stakes":   "Medical/legal/financial advice, incident response, risk assessment",
}

for name, strategies in REASONING_ARCHETYPES.items():
    combo = " + ".join(strategies)
    use = archetype_use_cases.get(name, "")
    md(f"  {name:<16} = {combo}")
    md(f"  {'':16}   Use when: {use}")


Named archetypes (pre-validated strategy compositions):


  analytical       = step_by_step + verify

                     Use when: Data analysis, structured problem-solving, reports

  deliberative     = multiple_angles + step_by_step + verify

                     Use when: Strategy decisions, trade-off analysis, policy questions

...

                     Use when: Documentation, tutorials, customer-facing explanations

  creative         = creative + multiple_angles

                     Use when: Brainstorming, ideation, copywriting, scenario planning

  high_stakes      = step_by_step + multiple_angles + verify

                     Use when: Medical/legal/financial advice, incident response, risk assessment

## Part 5 — Same task, different strategies: watch the prompt change

In [8]:
from mycontext import Context, Guidance, Directive, THINKING_STRATEGIES
from mycontext.intelligence import QualityMetrics

task = "Assess the risk of deploying a new ML model in our fraud detection pipeline"
qm = QualityMetrics(mode="heuristic")

# thinking_strategy only appears in assembled output when research_flow=True
# (same 9-section ordering as PromptArchitect). Context.thinking_strategy takes
# one atomic slug from THINKING_STRATEGIES (archetype names like high_stakes are for PromptArchitect JSON).
strategies_to_compare = [None, "step_by_step", "multiple_angles", "verify"]

results = []
for strategy in strategies_to_compare:
    ctx = Context(
        guidance=Guidance(
            role="ML Risk Analyst",
            rules=["Must identify at least 3 risk categories", "Must quantify each risk where possible"],
            goal="Assess deployment risk systematically",
        ),
        directive=Directive(content=task),
        thinking_strategy=strategy if strategy in THINKING_STRATEGIES else None,
        research_flow=True,
        provider_hint="openai",
    )
    score = qm.evaluate(ctx)
    assembled = ctx.assemble()
    results.append((strategy or "none", score.overall, len(assembled)))

md(f"{'Strategy':<20} {'Quality':>12} {'Prompt length (chars)'}")
md("-" * 58)
for strat, overall_01, length in results:
    q100 = overall_01 * 100
    md(f"{strat:<20} {q100:>10.1f}/100  {length} chars")


Strategy                  Quality Prompt length (chars)

----------------------------------------------------------

none                       63.6/100  367 chars

step_by_step               64.1/100  559 chars

multiple_angles            64.1/100  605 chars

verify                     64.1/100  592 chars

In [10]:
# Show exactly what the reasoning section looks like for each strategy
md("What each strategy injects (rendered in the recency zone before YOUR TASK, 0.10.1+):\n")
for slug, (label, injection) in THINKING_STRATEGIES.items():
    md(f"── {slug} ({label}) ──")
    md(f"```\n{injection}\n```")


What each strategy injects (rendered in the recency zone before YOUR TASK, 0.10.1+):


── step_by_step (Chain of Thought) ──

```
Think through this step by step. Break the problem down into stages, show your reasoning at each stage, then give your final answer.
```

── multiple_angles (Tree of Thought) ──

```
Before answering, brainstorm at least 3 different approaches or perspectives. Briefly evaluate the strengths and weaknesses of each, then choose the best approach and explain why.
```

...

```
After providing your answer, critically review it. Check for errors, missing information, unsupported claims, or logical gaps. If you find issues, revise your answer.
```

── explain_simply (Simplification) ──

```
Explain your reasoning in simple, everyday language that anyone can understand. Avoid jargon and technical terms. Use analogies where helpful.
```

── creative (Divergent Thinking) ──

```
Explore unconventional, surprising, and creative ideas. Don't limit yourself to the obvious answer. Challenge assumptions and consider perspectives that others might miss.
```

## Part 6 — `build()` with explicit strategy control

In [ ]:
# Build with deliberative archetype — best for high-stakes strategic decisions
# Re-create architect here so this cell runs independently of Part 1
from mycontext.intelligence import PromptArchitect
architect = PromptArchitect(provider="openai")

result_deliberative = architect.build(
    task="Assess the risk of deploying a new ML model in our fraud detection pipeline",
    reasoning_strategies=["deliberative"],  # expands to: multiple_angles + step_by_step + verify
)
md("### Built with `deliberative` archetype")
_ad = str(result_deliberative.improved_context.assemble())
_ad = _ad[:1497] + "..." if len(_ad) > 1500 else _ad
md(f"```\n{_ad}\n```")
md(f"**Score lift:** +{result_deliberative.score_delta:.0%} over baseline")


## Part 7 — Provider-aware rendering: same prompt, different structure

When you set `provider_hint`, the same 9-section Context renders differently — because different models respond better to different formatting conventions. This is backed by the provider-specific prompt rendering research in the mycontext knowledge base.

The loop below uses `ctx.model_copy(update={"provider_hint": ...})` so you preview OpenAI vs Anthropic vs Gemini skins without mutating the original `Context`. (`assemble_for_model` is for token-budget trimming with a **model** name, not for this provider-style pass.)

In [ ]:
from mycontext import Context, Guidance, Directive, Constraints

ctx = Context(
    guidance=Guidance(
        role="Senior data analyst specialising in fraud detection",
        rules=[
            "Must identify at least 3 distinct risk categories",
            "Must quantify each risk with a probability or severity score",
            "Must cite the model metric (precision/recall) affected by each risk",
        ],
        goal="Deliver a deployment risk assessment the engineering team can act on",
        style="Structured, evidence-based, no hedge language",
    ),
    directive=Directive(content="Assess the risk of deploying a new ML model in our fraud detection pipeline"),
    constraints=Constraints(format_rules=["Use numbered sections", "End with a GO/NO-GO recommendation"]),
    thinking_strategy="verify",
    research_flow=True,
)

for provider_hint in ("openai", "anthropic", "gemini"):
    skin_ctx = ctx.model_copy(update={"provider_hint": provider_hint})
    rendered = skin_ctx.assemble()
    _rv = str(rendered)
    _rv = _rv[:597] + "..." if len(_rv) > 600 else _rv
    md(f"### Provider `{provider_hint}` — {len(rendered)} chars\n\n```\n{_rv}\n```")


## Part 8 — The linguistic rules baked into every generated section

`build()` doesn't just fill sections — it applies 13 linguistic rules distilled from prompt engineering research to every word it writes.

In [ ]:
# These rules are embedded in every LLM rewriting call inside PromptArchitect
linguistic_rules = [
    ("Imperative framing",   "Goals as directives, not descriptions",                       "'Analyze' not 'You will analyze'"),
    ("Binding modals",       "must / always / never / shall — never 'should' for constraints", "'Must cite source' not 'Should cite source'"),
    ("Specificity",          "Every rule must be objectively testable",                     "'≥3 data points' not 'be detailed'"),
    ("Positive redirect",    "Never bare negation — state positive action + fallback",       "'If absent, state: NOT FOUND' not 'Don't make things up'"),
    ("One rule = one sentence", "Split compound rules",                                     "Prevents ambiguity in constraint parsing"),
    ("Task last",            "Final instruction always in recency zone",                    "Highest recall at generation start"),
    ("Grounding protocol",   "Claims must be traceable to provided material",               "Prevents hallucination in knowledge-grounded tasks"),
    ("Anti-boilerplate",     "Suppress 'delve into', 'it's no wonder' etc.",               "Forces domain-practitioner voice"),
]

md(f"{'Rule':<25} {'Principle':<48} {'Example'}")
md("-" * 110)
for rule, principle, example in linguistic_rules:
    md(f"{rule:<25} {principle:<48} {example}")


## Summary

| What you saw | API |
|-------------|-----|
| Task string → full 9-section prompt | `PromptArchitect.build(task)` |
| 9-section schema vs assembled order | `SECTION_NAMES` + **assembly_guide** table above |
| 5 atomic thinking strategies | `THINKING_STRATEGIES` dict |
| 5 named archetypes (strategy combos) | `REASONING_ARCHETYPES` dict |
| Archetype in build() | `architect.build(task, reasoning_strategies=['deliberative'])` |
| Provider-aware rendering | `ctx.model_copy(update={'provider_hint': hint}).assemble()` |
| 13 embedded linguistic rules | Automatic — applied in every `build()` call |

**Next:** [02_prompt_architect_improve.ipynb](02_prompt_architect_improve.ipynb) — when you have an existing prompt that needs diagnosis and repair, or you want to build manually and improve iteratively.